# Temporal Data Splitting

Splits graph tensors chronologically into Train (60%), Validation (20%), and Test (20%) sets.


## 1. Load Graph Tensors


In [2]:
import os
import torch

graph_path = "graph_data.pt"
if not os.path.exists(graph_path):
    graph_path = os.path.join("model", "graph_data.pt")

graph_data = torch.load(graph_path, weights_only=False)
x = graph_data["x"]
edge_index = graph_data["edge_index"]
edge_attr = graph_data["edge_attr"]
timestamps = graph_data["timestamps"]
y = graph_data["y"]
print(f"Loaded graph data: {x.shape[0]:,} nodes, {edge_index.shape[1]:,} edges")


Loaded graph data from graph_data.pt:
  x          : torch.Size([1754264, 1])
  edge_index : torch.Size([2, 5000000])
  edge_attr  : torch.Size([5000000, 4])
  timestamps : torch.Size([5000000])
  y          : torch.Size([5000000]) (Illicit ratio: 0.056%)


## 2. Bucket Transactions into Days


In [3]:
import numpy as np
import itertools

n_days = int(timestamps.max() / (3600 * 24) + 1)
daily_irs = []
daily_inds = []
daily_trans = []

for day in range(n_days):
    l = day * 24 * 3600
    r = (day + 1) * 24 * 3600
    day_inds = torch.where((timestamps >= l) & (timestamps < r))[0]
    daily_irs.append(y[day_inds].float().mean().item())
    daily_inds.append(day_inds)
    daily_trans.append(day_inds.shape[0])

print(f"Processed {n_days} calendar days")


Days processed : 69
Transactions per day (first 5 days) : [4466821, 530666, 74, 83, 85]
Illicit ratio per day (first 5 days) : [0.0002, 0.0006, 0.6216, 0.5783, 0.6118]


## 3. Find Optimal Split Cut-Points


In [4]:
split_per = [0.6, 0.2, 0.2]
daily_totals = np.array(daily_trans)
d_ts = daily_totals
I = list(range(len(d_ts)))
split_scores = {}

for i, j in itertools.combinations(I, 2):
    if j >= i:
        split_totals = [d_ts[:i].sum(), d_ts[i:j].sum(), d_ts[j:].sum()]
        split_totals_sum = np.sum(split_totals)
        split_props = [v / split_totals_sum for v in split_totals]
        split_error = [abs(v - t) / t for v, t in zip(split_props, split_per)]
        score = max(split_error)
        split_scores[(i, j)] = score

best_i, best_j = min(split_scores, key=split_scores.get)
split = [
    list(range(best_i)),
    list(range(best_i, best_j)),
    list(range(best_j, len(daily_totals)))
]
print(f"Optimal cut-points: Day {best_i} and Day {best_j}")


Optimal cut-points : Day 1 and Day 2
Train days : Day 0 to Day 0  (1 days)
Validation days : Day 1 to Day 1  (1 days)
Test days : Day 2 to Day 68  (67 days)


## 4. Collect Transaction Indices


In [6]:
split_inds = {k: [] for k in range(3)}
for k in range(3):
    for day in split[k]:
        split_inds[k].append(daily_inds[day])

tr_inds = torch.cat(split_inds[0])
val_inds = torch.cat(split_inds[1])
te_inds = torch.cat(split_inds[2])
total = y.shape[0]
print(f"Train: {tr_inds.shape[0]:,} ({tr_inds.shape[0]/total*100:.1f}%), Val: {val_inds.shape[0]:,} ({val_inds.shape[0]/total*100:.1f}%), Test: {te_inds.shape[0]:,} ({te_inds.shape[0]/total*100:.1f}%)")


Train transactions: 4466821 (89.3%) || Illicit : 0.023%
Val transactions: 530666 (10.6%) || Illicit : 0.061%
Test transactions: 2513 (0.1%) || Illicit : 58.456%


## 5. Create PyG Data Objects


In [7]:
from torch_geometric.data import Data

tr_x = val_x = te_x = x
e_tr = tr_inds.numpy()
e_val = np.concatenate([tr_inds.numpy(), val_inds.numpy()])

tr_edge_index = edge_index[:, e_tr]
tr_edge_attr = edge_attr[e_tr]
tr_y = y[e_tr]
tr_edge_times = timestamps[e_tr]

val_edge_index = edge_index[:, e_val]
val_edge_attr = edge_attr[e_val]
val_y = y[e_val]
val_edge_times = timestamps[e_val]

te_edge_index = edge_index
te_edge_attr = edge_attr
te_y = y
te_edge_times = timestamps

tr_data = Data(x=tr_x, edge_index=tr_edge_index, edge_attr=tr_edge_attr, y=tr_y)
val_data = Data(x=val_x, edge_index=val_edge_index, edge_attr=val_edge_attr, y=val_y)
te_data = Data(x=te_x, edge_index=te_edge_index, edge_attr=te_edge_attr, y=te_y)

tr_data.timestamps = tr_edge_times
val_data.timestamps = val_edge_times
te_data.timestamps = te_edge_times
print(f"Created Data objects: Train ({tr_data.num_edges:,} edges), Val ({val_data.num_edges:,} edges), Test ({te_data.num_edges:,} edges)")


Train Data : Data(x=[1754264, 1], edge_index=[2, 4466821], edge_attr=[4466821, 4], y=[4466821], timestamps=[4466821])
Val Data : Data(x=[1754264, 1], edge_index=[2, 4997487], edge_attr=[4997487, 4], y=[4997487], timestamps=[4997487])
Test Data : Data(x=[1754264, 1], edge_index=[2, 5000000], edge_attr=[5000000, 4], y=[5000000], timestamps=[5000000])


## 6. Normalize Features


In [8]:
def z_norm(data):
    std = data.std(0).unsqueeze(0)
    std = torch.where(std == 0, torch.tensor(1, dtype=torch.float), std)
    return (data - data.mean(0).unsqueeze(0)) / std

tr_data.x = val_data.x = te_data.x = z_norm(tr_data.x)
tr_data.edge_attr = z_norm(tr_data.edge_attr)
val_data.edge_attr = z_norm(val_data.edge_attr)
te_data.edge_attr = z_norm(te_data.edge_attr)
print("Features normalized via Z-score")


Edge attr after z-norm (train, row 0) : [-1.103341817855835, -0.002066870918497443, 1.6104360818862915, 1.085208535194397]
Edge attr mean (should be ~0) : [-2.1119255677604087e-07, 7.081854547230648e-10, -4.859642643850748e-08, -1.0658010030084597e-09]
Edge attr std (should be ~1) : [1.0, 1.0, 1.0, 0.9999999403953552]


## 7. Save Split Data Objects


In [ ]:
save_path = "split_data.pt"
torch.save({"tr_data": tr_data, "val_data": val_data, "te_data": te_data, "tr_inds": tr_inds, "val_inds": val_inds, "te_inds": te_inds}, save_path)
print(f"Saved split_data.pt -> {save_path} ({os.path.getsize(save_path) / 1e9:.2f} GB)")
